# 1. Trace Framework and DFA Initialization

This notebook introduces the full **OpSymbol alphabet** and **TraceStep** dataclass
used by the NeuroGolf multi-agent ONNX solver pipeline, then demonstrates the
**DFA verifier** that validates execution traces against the formal state machine.


In [2]:
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Optional


In [3]:
class OpSymbol(Enum):
    ANALYZE_TASK = auto()
    DISCOVER_PATTERN = auto()
    BUILD_ONNX = auto()
    ENCODE_RULE = auto()
    LABEL_PROPAGATE = auto()
    CONVOLUTION = auto()
    SCATTERND_HIST = auto()
    FP16_SURGERY = auto()
    CAST_COLLAPSE = auto()
    REDUCE_FUSION = auto()
    DTYPE_NARROW = auto()
    PRUNE = auto()
    GRAPH_REWRITE = auto()
    DIM_SCRUB = auto()
    VERIFY_TRAIN = auto()
    VERIFY_TEST = auto()
    VERIFY_ARC_GEN = auto()
    K_FOLD_CV = auto()
    HYPOTHESIS_TEST = auto()
    DATA_VALIDATION = auto()
    EARLY_STOPPING = auto()
    MODEL_GOVERNANCE = auto()
    COMPUTE_COST = auto()
    COST_GRADER_MATCH = auto()
    DISCOVER_BUNDLE = auto()
    LOAD_FLOOR = auto()
    BLEND_BUNDLE = auto()
    SHA256_CHECK = auto()
    SIZE_AUDIT = auto()
    PACKAGE_SUBMISSION = auto()
    SUBMIT = auto()
    REJECT = auto()
    HALT = auto()
    AUTO_ML = auto()
    MCTS_SEARCH = auto()
    SELF_ATTENTION = auto()
    FEW_SHOT_LEARNING = auto()
    DATA_AUGMENTATION = auto()
    HYPERPARAM_OPT = auto()
    ENSEMBLE_LEARNING = auto()
    TRANSFER_LEARNING = auto()

print(f"Defined {len(OpSymbol)} operation symbols for the trace language.")


Defined 41 operation symbols for the trace language.


In [4]:
@dataclass
class TraceStep:
    op: OpSymbol
    agent: str
    task_id: Optional[int] = None
    detail: str = ""
    metadata: dict = field(default_factory=dict)

# Demonstrate a few steps
steps = [
    TraceStep(OpSymbol.DISCOVER_BUNDLE, "scanner"),
    TraceStep(OpSymbol.LOAD_FLOOR, "scanner"),
    TraceStep(OpSymbol.ANALYZE_TASK, "analyzer", task_id=1),
    TraceStep(OpSymbol.BUILD_ONNX, "builder", task_id=1,
              detail="make_kronecker_tile"),
    TraceStep(OpSymbol.FP16_SURGERY, "optimizer", task_id=1),
    TraceStep(OpSymbol.VERIFY_TRAIN, "verifier", task_id=1),
    TraceStep(OpSymbol.COMPUTE_COST, "grader", task_id=1),
    TraceStep(OpSymbol.BLEND_BUNDLE, "blender"),
    TraceStep(OpSymbol.PACKAGE_SUBMISSION, "packager"),
    TraceStep(OpSymbol.SUBMIT, "orch"),
]

print(f"Constructed a {len(steps)}-step pipeline trace:")
for s in steps:
    tid = f"task={s.task_id}" if s.task_id is not None else ""
    d = f"  [{s.detail}]" if s.detail else ""
    print(f"  {s.agent:>10s} | {s.op.name:20s} {tid}{d}")


Constructed a 10-step pipeline trace:
     scanner | DISCOVER_BUNDLE      
     scanner | LOAD_FLOOR           
    analyzer | ANALYZE_TASK         task=1
     builder | BUILD_ONNX           task=1  [make_kronecker_tile]
   optimizer | FP16_SURGERY         task=1
    verifier | VERIFY_TRAIN         task=1
      grader | COMPUTE_COST         task=1
     blender | BLEND_BUNDLE         
    packager | PACKAGE_SUBMISSION   
        orch | SUBMIT               


## DFA Verification

The real DFA verifier (`framework.neurogolf_dfa_verifier.verify_trace`)
implements a state machine with 13 states and 100+ transitions.
Let's demonstrate it:


In [5]:
import sys, os, glob
if os.path.exists("../../framework"):
    sys.path.insert(0, "../../")
elif os.path.exists("../../../framework"):
    sys.path.insert(0, "../../../")
else:
    _k_paths = glob.glob("~/kaggle/input/*/framework")
    if _k_paths:
        sys.path.insert(0, os.path.dirname(_k_paths[0]))
    else:
        sys.path.insert(0, ".")

from framework.neurogolf_dfa_verifier import (
    verify_trace, make_initial_trace, make_standard_pipeline, DFAState,
)

# Generate a standard pipeline trace for tasks 1..3
pipeline = make_standard_pipeline([1, 2, 3])
print(f"Pipeline has {len(pipeline)} steps.\n")

# Verify it against the DFA
result = verify_trace(pipeline, code_text="", min_optimizations=1)

print(f"Accepted:  {result.accepted}")
print(f"Final DFA: {result.final_state.name}")
print(f"Opts applied: {len(result.optimizations_applied)}")
print(f"Coverage ratio: {result.coverage_ratio:.0%}")
print(f"Errors: {len(result.errors)}")
if result.errors:
    for e in result.errors[:3]:
        print(f"  ! {e}")


Pipeline has 29 steps.

Accepted:  True
Final DFA: SUBMITTED
Opts applied: 1
Coverage ratio: 0%
Errors: 0


In [6]:
# Show the first few transition steps
print("Step-by-step transitions (first 8):")
for i, (idx, step, ok, msg) in enumerate(result.step_results[:8]):
    marker = "\u2713" if ok else "\u2717"
    print(f"  [{idx}] {marker} {step.agent:>10s} {step.op.name:20s}  {msg}")


Step-by-step transitions (first 8):
  [0] ✓    scanner DISCOVER_BUNDLE       -> INIT
  [1] ✓    scanner LOAD_FLOOR            -> INIT
  [2] ✓   analyzer ANALYZE_TASK          -> ANALYZED
  [3] ✓    builder BUILD_ONNX            -> BUILT
  [4] ✓  optimizer FP16_SURGERY          -> OPTIMIZED
  [5] ✓  optimizer FP16_SURGERY          -> OPTIMIZED
  [6] ✓   verifier VERIFY_TRAIN          -> V_TRAIN
  [7] ✓   verifier VERIFY_TEST           -> V_TEST
